# โครงการวิเคราะห์ข้อมูลผู้กระทำผิดคดีคุมความประพฤติ
ภาพรวมและวัตถุประสงค์: เพื่อวิเคราะห์แนวโน้มและลักษณะของการรับคดีคุมความประพฤติผู้ใหญ่ ทั้งในระดับพื้นที่และประเภทฐานความผิด

## ขั้นตอนที่ 0: นำเข้าไลบรารีและการตั้งค่าเบื้องต้น

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

## กรมคุมประพฤติ กระทรวงยุติธรรม: การรับคดีคุมความประพฤติผู้กระทำผิดที่เป็นผู้ใหญ่
คำอธิบาย: ข้อมูลจำนวนคดีสอดส่องผู้ใหญ่ จำแนกตามจังหวัดและฐานความผิด ปี 2565-2566
แหล่งที่มา: https://gdcatalog.go.th/dataset/gdpublish-probation-21-01

In [2]:
# Load the data
df_combined = pd.read_csv('../data/processed/probation_adult_combined.csv')
df_guilty = pd.read_csv('../data/processed/probation_adult_2565_guilty_clean.csv')

display(df_combined.head())
display(df_guilty.head())

,สำนักงานคุมประพฤติ,เดือน ต.ค.2564,เดือน พ.ย.2564,เดือน ธ.ค.2564,เดือน ม.ค.2565,เดือน ก.พ.2565,เดือน มี.ค.2565,เดือน เม.ย.2565,เดือน พ.ค.2565,เดือน มิ.ย.2565,...,เดือน ธ.ค.2565,เดือน ม.ค.2566,เดือน ก.พ.2566,เดือน มี.ค.2566,เดือน เม.ย.2566,เดือน พ.ค.2566,เดือน มิ.ย.2566,เดือน ก.ค.2566,เดือน ส.ค.2566,เดือน ก.ย.2566
0,สำนักงานคุมประพฤติกรุงเทพมหานคร 1,27.0,41.0,54.0,33.0,31.0,38.0,58.0,35.0,43.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,สำนักงานคุมประพฤติกรุงเทพมหานคร 2,30.0,45.0,145.0,178.0,47.0,68.0,253.0,119.0,161.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,สำนักงานคุมประพฤติกรุงเทพมหานคร 3,27.0,31.0,37.0,77.0,40.0,65.0,126.0,133.0,187.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,สำนักงานคุมประพฤติกรุงเทพมหานคร 4,36.0,59.0,88.0,203.0,133.0,221.0,245.0,244.0,308.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,สำนักงานคุมประพฤติกรุงเทพมหานคร 5,4.0,8.0,8.0,10.0,18.0,7.0,12.0,10.0,17.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,ฐานความผิด,เดือน ต.ค.2564,เดือน พ.ย.2564,เดือน ธ.ค.2564,เดือน ม.ค.2565,เดือน ก.พ.2565,เดือน มี.ค.2565,เดือน เม.ย.2565,เดือน พ.ค.2565,เดือน มิ.ย.2565,เดือน ก.ค.2565,เดือน ส.ค.2565,เดือน ก.ย.2565,year_be
0,ความผิดเกี่ยวกับทรัพย์,405,423,487,440,433,609,444,487,529,473,627,544,2565
1,ความผิดเกี่ยวกับเพศ,32,35,36,45,50,69,24,43,47,54,61,56,2565
2,ความผิดเกี่ยวกับเอกสาร,39,37,39,35,35,57,18,37,39,34,45,51,2565
3,ความผิดฐานขับรถประมาท,490,560,534,435,530,692,439,489,574,448,583,556,2565
4,ความผิดฐานบุกรุก,68,102,125,103,115,145,99,101,141,91,161,119,2565


ปัญหาของข้อมูลและวิธีการทำความสะอาดข้อมูล (Data Cleaning)
- ชื่อสำนักงานคุมประพฤติในข้อมูลรวม มีการระบุสาขาหรือเลขเขต เช่น "กรุงเทพมหานคร 1" เราจะสกัดชื่อจังหวัดออกมาเพื่อนำไปใช้วิเคราะห์ระดับจังหวัด
- ข้อมูลอยู่ในรูปแบบ Wide Format (คอลัมน์เป็นเดือน) จะทำการ Melt ข้อมูลให้อยู่ในรูปแบบ Long Format เพื่อให้ง่ายต่อการวิเคราะห์ Time Series
- ดึงข้อมูลปีและเดือนเพื่อสร้างคอลัมน์เวลาและปีงบประมาณ (Fiscal Year)

In [3]:
# Clean province names
def extract_province(name):
    name = str(name).replace('สำนักงานคุมประพฤติ', '').replace('จังหวัด', '').strip()
    name = name.split(' ')[0]
    return name

df_combined['Province'] = df_combined['สำนักงานคุมประพฤติ'].apply(extract_province)

# Melt df_combined
id_vars = ['สำนักงานคุมประพฤติ', 'Province', 'year_be']
month_cols = [c for c in df_combined.columns if c not in id_vars]
df_long = df_combined.melt(id_vars=id_vars, value_vars=month_cols, var_name='Month_Year', value_name='Cases')

# Clean Cases
df_long['Cases'] = df_long['Cases'].fillna(0).astype(int)
df_long['Fiscal_Year'] = df_long['year_be']

month_map = {'ม.ค.': '01', 'ก.พ.': '02', 'มี.ค.': '03', 'เม.ย.': '04', 'พ.ค.': '05', 'มิ.ย.': '06',
             'ก.ค.': '07', 'ส.ค.': '08', 'ก.ย.': '09', 'ต.ค.': '10', 'พ.ย.': '11', 'ธ.ค.': '12'}

def parse_date(my):
    parts = str(my).split(' ')
    if len(parts) >= 2:
        m_y = parts[1]
        for th, en in month_map.items():
            if th in m_y:
                y = m_y.replace(th, '')
                return f"{int(y)-543}-{en}-01"
    return None

df_long['Date'] = pd.to_datetime(df_long['Month_Year'].apply(parse_date))
df_long = df_long.sort_values('Date')

# Clean df_guilty
id_vars_g = ['ฐานความผิด', 'year_be']
month_cols_g = [c for c in df_guilty.columns if c not in id_vars_g]
df_g_long = df_guilty.melt(id_vars=id_vars_g, value_vars=month_cols_g, var_name='Month_Year', value_name='Cases')
df_g_long['Cases'] = df_g_long['Cases'].fillna(0).astype(int)
df_g_long['Fiscal_Year'] = df_g_long['year_be']
df_g_long['Date'] = pd.to_datetime(df_g_long['Month_Year'].apply(parse_date))
df_g_long = df_g_long.sort_values('Date')


### แผนภาพที่ 1: อันดับจังหวัดที่รับคดีคุมความประพฤติสูงสุด
เป้าหมาย: แสดง 10 จังหวัดที่มีจำนวนคดีสอดส่องรวมสูงสุด

In [4]:
df_prov = df_long.groupby('Province')['Cases'].sum().reset_index().sort_values('Cases', ascending=False).head(10)

fig1 = px.bar(df_prov, x='Province', y='Cases', 
              title='Top 10 Provinces by Total Probation Cases (FY2565-2566)',
              labels={'Province': 'Province', 'Cases': 'Total Cases'},
              text_auto=True)
fig1.show()


**คำอธิบายแผนภาพที่ 1:** แผนภาพแสดง 10 จังหวัดที่มีจำนวนคดีสอดส่องสูงสุด

**คอลัมน์ที่ใช้:** สำนักงานคุมประพฤติ (แปลงเป็นจังหวัด), Cases

**การแปลงข้อมูล:** หาผลรวม (Sum)

**หน่วย:** คดี

### แผนภาพที่ 2: สัดส่วนคดีคุมความประพฤติจำแนกตามฐานความผิด
เป้าหมาย: แสดงสัดส่วนของแต่ละฐานความผิดแบบ Stacked Bar

In [5]:
df_g_agg = df_g_long.groupby(['Date', 'ฐานความผิด'])['Cases'].sum().reset_index()
df_g_agg['Date_Str'] = df_g_agg['Date'].dt.strftime('%Y-%m')

fig2 = px.bar(df_g_agg, x='Date_Str', y='Cases', color='ฐานความผิด',
              title='Probation Cases by Type of Offense over Time (FY2565)',
              labels={'Date_Str': 'Time (Fiscal Year 2565)', 'Cases': 'Total Cases', 'ฐานความผิด': 'Offense Type'},
              barmode='stack')
fig2.update_layout(xaxis_title='Time (Note: Data represents Fiscal Year Oct-Sep)')
fig2.show()


**คำอธิบายแผนภาพที่ 2:** แผนภาพแบบ Stacked Bar แสดงสัดส่วนคดีจำแนกตามฐานความผิด

**คอลัมน์ที่ใช้:** เดือน (แปลงเป็นวันที่), ฐานความผิด, Cases

**การแปลงข้อมูล:** หาผลรวมรายเดือน (Sum)

**หน่วย:** คดี

### แผนภาพที่ 2 (เพิ่มเติม): แนวโน้มคดีคุมความประพฤติ 5 อันดับแรก
เป้าหมาย: แสดงแนวโน้มของฐานความผิดหลักเพื่อให้เปรียบเทียบและการอ่านข้อมูลได้ง่ายและชัดเจนขึ้น

In [6]:
# หา 5 ฐานความผิดที่มีจำนวนคดีรวมสูงสุด
top_5_offenses = df_g_agg.groupby('ฐานความผิด')['Cases'].sum().nlargest(5).index
df_top5 = df_g_agg[df_g_agg['ฐานความผิด'].isin(top_5_offenses)]

fig2_1 = px.line(df_top5, x='Date_Str', y='Cases', color='ฐานความผิด', markers=True,
                 title='Top 5 Probation Cases by Type of Offense over Time (FY2565)',
                 labels={'Date_Str': 'Time (Fiscal Year 2565)', 'Cases': 'Total Cases', 'ฐานความผิด': 'Offense Type'})
fig2_1.update_layout(xaxis_title='Time (Note: Data represents Fiscal Year Oct-Sep)')
fig2_1.show()

**คำอธิบายแผนภาพที่ 2 (เพิ่มเติม):** แผนภาพ Line Chart แสดงแนวโน้มคดีจำแนกตามฐานความผิด 5 อันดับแรก (เพื่อให้เห็นการเปลี่ยนแปลงที่ชัดเจนและอ่านง่ายกว่าแบบ Stacked Bar)

**คอลัมน์ที่ใช้:** เดือน (แปลงเป็นวันที่), ฐานความผิด, Cases

**การแปลงข้อมูล:** คัดกรองเฉพาะ 5 ฐานความผิดสูงสุด และหาผลรวมรายเดือน (Sum)

**หน่วย:** คดี

### แผนภาพที่ 3: แนวโน้มจำนวนคดีคุมความประพฤติในแต่ละปีงบประมาณ
เป้าหมาย: วิเคราะห์แนวโน้มรวมของคดีตลอดปีงบประมาณ (Time Series)

In [7]:
df_time = df_long.groupby('Date')['Cases'].sum().reset_index()

fig3 = px.line(df_time, x='Date', y='Cases', markers=True,
               title='Total Probation Cases Trend (FY2565-2566)',
               labels={'Date': 'Time (Fiscal Year 2565-2566)', 'Cases': 'Total Cases'})
fig3.update_layout(xaxis_title='Time (Note: Data represents Fiscal Year Oct-Sep)')
fig3.show()


**คำอธิบายแผนภาพที่ 3:** แผนภาพ Time Series แสดงแนวโน้มคดีคุมความประพฤติรายปีงบประมาณ

**คอลัมน์ที่ใช้:** เดือน (แปลงเป็นวันที่), Cases

**การแปลงข้อมูล:** หาผลรวมรายเดือน (Sum)

**หน่วย:** คดี

### แผนภาพที่ 4: การจัดกลุ่มสำนักงานด้วย KMeans (Optional)
เป้าหมาย: จัดกลุ่มสำนักงานคุมประพฤติที่มีพฤติกรรมการรับคดีคล้ายคลึงกัน

In [8]:
df_kmeans = df_long.groupby('สำนักงานคุมประพฤติ').agg(
    Total_Cases=('Cases', 'sum'),
    Mean_Cases=('Cases', 'mean'),
    Std_Cases=('Cases', 'std')
).fillna(0).reset_index()

X = df_kmeans[['Total_Cases', 'Mean_Cases', 'Std_Cases']]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

kmeans = KMeans(n_clusters=3, random_state=42)
df_kmeans['Cluster'] = kmeans.fit_predict(X_scaled)
df_kmeans['Cluster'] = df_kmeans['Cluster'].astype(str)

fig4 = px.scatter(df_kmeans, x='Total_Cases', y='Std_Cases', color='Cluster',
                  hover_name='สำนักงานคุมประพฤติ',
                  title='KMeans Clustering of Probation Offices',
                  labels={'Total_Cases': 'Total Cases', 'Std_Cases': 'Standard Deviation of Cases'})
fig4.show()


**คำอธิบายแผนภาพที่ 4:** แผนภาพแสดงผลการทำ KMeans Clustering จัดกลุ่มสำนักงานคุมประพฤติ

**ฟีเจอร์ที่ใช้ (Features):** Total_Cases (ผลรวมจำนวนคดี), Mean_Cases (ค่าเฉลี่ยจำนวนคดี), Std_Cases (ส่วนเบี่ยงเบนมาตรฐานของจำนวนคดี)

**การแบ่งข้อมูล (Train/Test Split):** ไม่มี (ใช้การประมวลผลข้อมูลทั้งหมดแบบไม่มีผู้สอน)

**ตัวชี้วัด (Metric):** Euclidean Distance (ภายในอัลกอริทึม KMeans)

**ข้อจำกัด (Limitations):** เป็นการใช้ข้อมูลที่ผ่านการรวมกลุ่มแล้ว (Aggregate data) และมีขอบเขตครอบคลุมภาพรวมทั่วประเทศ

---

## สรุปผล
- จังหวัดใหญ่มีความหนาแน่นของคดีสูงที่สุด
- ฐานความผิดมีแนวโน้มการเปลี่ยนแปลงตามฤดูกาลและมีประเภทหลักที่มีสัดส่วนสูงสุด
- การจัดกลุ่ม KMeans สามารถแยกกลุ่มสำนักงานที่มีปริมาณคดีสูงออกจากกลุ่มทั่วไปได้อย่างชัดเจน